<div style='padding:20px; background-color:#232F3E; border-radius:10px; border-left: 10px solid #FF9900'>
<h1 style='color:#FF9900'>AWS ETL Pipeline Masterclass</h1>
<p style='color:white'>AWS Glue · Amazon S3 · Amazon Redshift · Batch vs. Streaming</p>
</div>

## 1. ETL vs. ELT – Conceptual Clarity

Choosing between ETL (Extract-Transform-Load) and ELT (Extract-Load-Transform) determines cost, latency, and compliance flexibility.



### **Comparison Matrix**
| Feature | **ETL** (Pre-process) | **ELT** (Post-process) |
| :--- | :--- | :--- |
| **Engine** | AWS Glue / Spark | Amazon Redshift / Snowflake |
| **Quality** | Cleaned *before* loading | Loaded raw, cleaned via SQL |
| **Compliance** | **High** (Mask PII before storage) | **Moderate** (Raw data sits in DB) |

> **Interview Tip:** ETL is optimized for **correctness** and regulatory compliance; ELT is optimized for **speed** and analytical flexibility.

## 2. Architecture: AWS Glue ➔ Redshift Pipeline

This standard batch processing architecture follows the Medallion pattern.



### **The Workflow**
1.  **Bronze Layer (Raw):** Immutable files (JSON/CSV) landed in S3.
2.  **Silver Layer (Curated):** Glue converts to **Parquet**, handles nulls, and deduplicates.
3.  **Gold Layer (Analytics):** Data loaded into Redshift for BI dashboards.

> **Pro Tip:** Always prefer **Parquet** for the Silver layer; it is columnar and significantly faster for Glue and Redshift Spectrum to process than JSON or CSV.

In [1]:
-- Idempotent Redshift Upsert (Merge Pattern)
BEGIN TRANSACTION;

-- 1. Bulk Load from S3 to Staging using COPY command
COPY flights_staging
FROM 's3://my-datalake/curated/flights/'
IAM_ROLE 'arn:aws:iam::123456789012:role/MyRedshiftRole'
FORMAT AS PARQUET;

-- 2. Delete existing records to handle updates
DELETE FROM flights_target
USING flights_staging
WHERE flights_target.flight_id = flights_staging.flight_id;

-- 3. Insert new records
INSERT INTO flights_target SELECT * FROM flights_staging;

END TRANSACTION;

SyntaxError: invalid syntax (205098184.py, line 1)

## 3. Reliability: Job Bookmarks

**Idempotency** ensures running the same job multiple times produces the same result. AWS Glue Job Bookmarks track processed state to ensure incremental ingestion.



* **Enabled:** Glue only processes *new* files since the last run.
* **Disabled:** Glue processes *all* files, useful for full historical backfills.

## 4. Redshift Performance: DistKeys & SortKeys

Performance is maximized by minimizing network shuffle (data movement across nodes).



| Style | Best Use Case |
| :--- | :--- |
| **KEY** | Large tables joined on a specific key (e.g., `customer_id`). |
| **ALL** | Small lookup/dimension tables to eliminate broadcast traffic. |
| **EVEN** | Round-robin distribution for tables that don't participate in frequent joins. |

## 5. Batch vs. Streaming Trade-offs

| Feature | **Batch** (S3 + Glue) | **Streaming** (Kinesis/Kafka) |
| :--- | :--- | :--- |
| **Latency** | Hours / Daily | Seconds / Milliseconds |
| **Complexity** | Low (File-based) | High (Event-based) |
| **Correction** | Easy (Reprocess file) | Hard (Replay offsets) |

### **Kinesis vs. Kafka**
* **Kinesis:** Fully managed, native integration with AWS Lambda/Firehose.
* **Kafka:** Open-source standard (managed via MSK); better for high-throughput or multi-cloud strategies.

<div style='padding:20px; background-color:#111; border-radius:10px; border-left: 10px solid #FF9900'>
<h1 style='color:#FF9900'>Data Lake & Warehouse Architecture</h1>
<p style='color:white'>Medallion Architecture | Star Schema | SCD Types | Partitioning</p>
</div>

## 1. The Medallion Architecture (Data Lakehouse)
The Medallion architecture is a data design pattern used to incrementally improve data quality.



### **Layer Definitions:**
* **Bronze (Raw):** Landing zone for source data. Append-only, immutable, and raw format.
* **Silver (Validated):** Cleaned, deduplicated, and normalized. The 'Source of Truth'.
* **Gold (Aggregated):** Business-ready data optimized for BI tools and ML.

## 2. Dimensional Modeling: Star vs. Snowflake
In modern warehouses like Redshift, the goal is to balance data integrity with query performance.



### **Star Schema (Best for Performance)**
* **Fact Table:** Quantitative metrics (Sales, Temperature).
* **Dimension Tables:** Descriptive attributes (Customer Name, Product Category).
* **Optimization:** Fewer joins lead to faster scans in columnar databases.

## 3. Slowly Changing Dimensions (SCD)

### **SCD Type 1: Overwrite**
Replaces the old value with the new value. No historical tracking.

### **SCD Type 2: Row Versioning**
Adds a new row with a timestamp to track history over time.



## 4. Partitioning & Redshift Distribution

### **S3 Partitioning**
Structure: `s3://bucket/table/year=2025/month=01/day=20/` to enable Partition Pruning.

### **Redshift Distribution Styles**


| Style | Use Case |
| :--- | :--- |
| **KEY** | Large tables joined on a specific key. |
| **ALL** | Small dimension tables to prevent network shuffle. |
| **EVEN** | Tables without a common join key. |

In [ ]:
-- Senior-Level SQL Architecture Example
CREATE TABLE dim_customer (
    cust_id INT,
    name VARCHAR(100),
    city VARCHAR(50)
) DISTSTYLE ALL;

CREATE TABLE fact_sales (
    order_id INT,
    cust_id INT,
    amount DECIMAL(10,2),
    sale_date DATE
) 
DISTSTYLE KEY DISTKEY(cust_id) 
SORTKEY(sale_date);

<div style='padding:20px; background-color:#2c3e50; border-radius:10px; border-left: 10px solid #27ae60'>
<h1 style='color:white'>Python & SQL for Data Engineering</h1>
<p style='color:#ecf0f1'>Advanced SQL · Memory Optimization · Data Quality · Unit Testing</p>
</div>

## 1. Advanced SQL: Deduplication with Window Functions

The most common use of Window Functions in ETL is **Deduplication**. We use `ROW_NUMBER()` to assign an index to rows within a partition, ordering by the most recent timestamp.



### **The Scenario**
We have a transaction table where `transaction_id` should be unique, but due to upstream retries, we have duplicates. We want to keep only the latest record.

In [ ]:
-- SQL Deduplication Pattern (Standard ANSI SQL / Redshift / Snowflake)

WITH ranked_transactions AS (
    SELECT 
        transaction_id,
        customer_id,
        amount,
        updated_at,
        -- Assign 1 to the latest row, 2+ to older duplicates
        ROW_NUMBER() OVER(
            PARTITION BY transaction_id 
            ORDER BY updated_at DESC
        ) as rn
    FROM raw_transactions
)
SELECT 
    transaction_id,
    customer_id,
    amount,
    updated_at
FROM ranked_transactions
WHERE rn = 1; -- Keep only the latest version

## 2. Python Optimization: Generators vs. Lists

When processing large datasets (e.g., 50GB logs) on a single EC2 instance, loading everything into a list will cause an **OOM (Out of Memory)** error.

### **The Solution: Generators**
Generators use `yield` to process one item at a time, keeping memory usage constant regardless of file size.



In [2]:
import sys

# BAD Practice: Loading everything into memory
def get_squares_list(n):
    return [x**2 for x in range(n)] # Creates a massive list in RAM

# GOOD Practice: Using a Generator
def get_squares_gen(n):
    for x in range(n):
        yield x**2 # Yields one number at a time

# Comparison
n = 1000000
print(f"List Size: {sys.getsizeof(get_squares_list(n))} bytes")
print(f"Generator Size: {sys.getsizeof(get_squares_gen(n))} bytes")

# The generator size will be tiny (~100 bytes) even if n is 1 billion.

List Size: 8448728 bytes
Generator Size: 200 bytes


## 3. Handling Bad Records (Dead Letter Queue)

In production pipelines, we never let one bad record crash the entire job. We separate data into **Valid** and **Invalid** streams.



In [3]:
data = [
    {"id": 1, "price": 100}, 
    {"id": 2, "price": "invalid"},  # Bad Record
    {"id": 3, "price": 50}
]

valid_records = []
bad_records = []

for record in data:
    try:
        # Validation Logic
        clean_price = float(record['price'])
        record['price'] = clean_price
        valid_records.append(record)
    except ValueError as e:
        # Capture the bad record and the error reason
        record['error'] = str(e)
        bad_records.append(record)

print("Valid:", valid_records)
print("To DLQ:", bad_records)

Valid: [{'id': 1, 'price': 100.0}, {'id': 3, 'price': 50.0}]
To DLQ: [{'id': 2, 'price': 'invalid', 'error': "could not convert string to float: 'invalid'"}]


## 4. Unit Testing ETL Code

We use `pytest` to verify transformation logic logic *before* deploying to production.



In [4]:
# Logic to be tested
def transform_currency(amount, rate):
    if amount < 0:
        raise ValueError("Negative amount")
    return round(amount * rate, 2)

# The Unit Test (usually in test_etl.py)
import pytest

def test_currency_conversion():
    assert transform_currency(100, 1.2) == 120.0
    assert transform_currency(10.555, 1) == 10.56

def test_negative_value():
    with pytest.raises(ValueError):
        transform_currency(-50, 1.2)

ModuleNotFoundError: No module named 'pytest'

<div style='padding:20px; background-color:#34495e; border-radius:10px; border-left: 10px solid #e67e22'>
<h1 style='color:white'>Automation & Orchestration</h1>
<p style='color:#ecf0f1'>Apache Airflow · AWS Step Functions · Retries · Error Handling</p>
</div>

## 1. Apache Airflow: Robust DAG Design

Airflow is a programmatic platform to author, schedule, and monitor workflows. A production DAG must include **retries**, **delays**, and **SLA alerts** to be reliable.



### **Key Parameters for Reliability:**
* `retries`: How many times to attempt a task before failing.
* `retry_delay`: How long to wait between attempts (Exponential Backoff is best).
* `on_failure_callback`: A function to trigger alerts (e.g., Slack or Email) when a task dies.

In [ ]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

def notify_failure(context):
    print(f"Task Failed! Sending alert for: {context['task_instance_key_str']}")

default_args = {
    'owner': 'data_eng',
    'depends_on_past': False,
    'start_date': datetime(2025, 1, 1),
    'retries': 3,
    'retry_delay': timedelta(minutes=5),
    'on_failure_callback': notify_failure
}

with DAG('enterprise_etl_v1', default_args=default_args, schedule_interval='@daily') as dag:
    
    extract = PythonOperator(
        task_id='extract_from_s3',
        python_callable=lambda: print("Extracting data...")
    )

    transform = PythonOperator(
        task_id='transform_data',
        python_callable=lambda: print("Transforming...")
    )

    extract >> transform

## 2. Step Functions vs. Airflow

Choosing the right tool depends on your infrastructure and the complexity of your logic.



| Feature | **AWS Step Functions** | **Apache Airflow (MWAA)** |
| :--- | :--- | :--- |
| **Type** | Serverless / State Machine | Managed Service (Python-based) |
| **Use Case** | Short-lived Lambda microservices | Complex Data Engineering pipelines |
| **Cost** | Per state transition | Flat hourly rate for environment |
| **Control** | Visual, JSON-based (ASL) | Full Python flexibility |
| **Scaling** | Instant / Automatic | Based on Worker nodes |

## 3. Handling Partial Failures

A pipeline shouldn't be "all or nothing." If 1 file out of 100 fails, the other 99 should proceed. This is handled via **Task Atomicity** and **Sub-DAGs/Dynamic Task Mapping**.



### **Strategies:**
1. **Checkpoints:** Save progress to S3 after every step.
2. **Idempotent Retries:** Ensure that if a task is halfway done and fails, rerunning it doesn't duplicate data.
3. **Soft Failures:** Use `AirflowSkipException` to skip specific tasks without failing the entire DAG.

<div style='padding:20px; background-color:#232F3E; border-radius:10px; border-left: 10px solid #FF9900'>
<h1 style='color:#FF9900'>Machine Learning for Data Engineers (10%)</h1>
<p style='color:white'>Supervised Learning · Anomaly Detection · Evaluation Metrics · AWS SageMaker</p>
</div>

## 1. Machine Learning in the Data Stack

For Data Engineers, Machine Learning is often the 'consumer' of the Gold Layer in a Medallion architecture. While Data Scientists build models, Data Engineers are responsible for the **ML Ops** lifecycle: feature stores, training pipelines, and endpoint scaling.



### **The Role of ML in Pipelines:**
* **Ingestion:** Moving raw data into S3/Data Lakes.
* **Feature Engineering:** Using Glue/Spark to transform raw data into tensors or vectors.
* **Inference:** Writing model predictions back to Redshift for BI consumption.

## 2. Classification vs. Regression

Understanding the output type determines the algorithm and evaluation strategy.



| Feature | **Classification** | **Regression** |
| :--- | :--- | :--- |
| **Output Type** | Discrete Class Labels | Continuous Numeric Values |
| **Use Case** | Fraud / Not Fraud | Predicted Sale Price |
| **Metric** | Accuracy, Precision, Recall | RMSE, MAE, R² |
| **Algorithms** | Logistic Regression, Random Forest | Linear Regression, XGBoost |

## 3. Anomaly Detection

Identifying 'Outliers' that deviate from normal behavior. Critical for cybersecurity and financial compliance.



### **Techniques:**
* **Statistical:** Z-Score and Interquartile Range (IQR).
* **ML-Based:** Isolation Forest and One-Class SVM.
* **Real-world Example:** Detecting a $5,000 credit card transaction when the user's average spend is $20.

## 4. Evaluation Metrics & Confusion Matrix

A model is only useful if it is accurate. We use a **Confusion Matrix** to break down errors.



### **Precision vs. Recall (The Trade-off):**
* **Precision:** "Of all items we called Positive, how many were actually correct?" (Low FP is the goal).
* **Recall:** "Of all actual Positives, how many did we successfully find?" (Low FN is the goal).

> **Interview Tip:** For Fraud Detection, **Recall** is more important (we'd rather flag a legitimate purchase than miss a fraudulent one). For Spam Detection, **Precision** is more important (we don't want real emails going to the spam folder).

## 5. Deploying with AWS SageMaker

SageMaker is the primary tool for productionizing ML on AWS.



### **Standard Deployment Workflow:**
1. **Store Data:** Training data resides in **Amazon S3**.
2. **Train:** SageMaker pulls data into an ephemeral training cluster.
3. **Evaluate:** Model artifacts are validated and stored back in S3.
4. **Deploy:** Create a **SageMaker Endpoint** for real-time inference or use **Batch Transform** for bulk predictions.

In [ ]:
# Conceptual SageMaker (boto3) Deployment
import boto3

def deploy_model(model_name, s3_artifact):
    sagemaker = boto3.client('sagemaker')
    
    # 1. Create Model
    sagemaker.create_model(
        ModelName=model_name,
        PrimaryContainer={'Image': '123.dkr.ecr.us-east-1.amazonaws.com/xgboost:latest', 'ModelDataUrl': s3_artifact}
    )
    
    # 2. Create Endpoint Configuration
    sagemaker.create_endpoint_config(
        EndpointConfigName=f"{model_name}-config",
        ProductionVariants=[{'VariantName': 'AllTraffic', 'ModelName': model_name, 'InstanceType': 'ml.m5.large', 'InitialInstanceCount': 1}]
    )
    
    print("Endpoint configuration created. Ready for deployment.")

<div style='padding:20px; background-color:#1a5276; border-radius:10px; border-left: 10px solid #f1c40f'>
<h1 style='color:white'>Analytics, Security & Governance</h1>
<p style='color:#d5dbdb'>Self-Service BI · PII Masking · Row-Level Security · Data Quality</p>
</div>

## 1. Self-Service Analytics Design

The goal of self-service is to allow non-technical users to build their own dashboards without writing SQL. In the AWS stack, this is powered by **Amazon QuickSight** and **QuickSight Q** (Natural Language Query).



### **Design Principles:**
* **Semantic Layer:** Rename technical column names (e.g., `cust_t_01`) to business terms (`Customer Name`).
* **Pre-aggregation:** Use the **SPICE engine** (Super-fast, Parallel, In-memory Calculation Engine) to provide sub-second response times.
* **Standardized Metrics:** Define 'Revenue' or 'Churn' once in the data warehouse so every dashboard shows the same number.

## 2. Row-Level Security (RLS) Implementation

RLS ensures that a manager in 'London' only sees London data, while a manager in 'New York' only sees New York data, even if they use the same dashboard.



### **How to implement in AWS:**
1. **Redshift:** Use `CREATE RLS POLICY` to restrict access based on user session context.
2. **QuickSight:** Create a 'User-to-Value' mapping file (CSV or Database table) that defines which user can see which region/tag.

## 3. PII Masking & Data Security

Personally Identifiable Information (PII) must be protected via encryption and masking.



### **Strategies:**
* **Encryption at Rest:** Use AWS KMS (Key Management Service) with S3 and Redshift.
* **Dynamic Masking:** Redshift can mask columns (e.g., `XXXX-XXXX-1234`) based on the user's IAM role.
* **Hashing:** Convert emails into irreversible hashes (SHA-256) for analytical tracking without exposing the identity.

## 4. Data Quality Framework (Glue DataBrew / Deequ)

A pipeline is only as good as the data it produces. We implement quality checks at the **Silver Layer** of the Medallion architecture.



### **Standard Quality Checks:**
* **Completeness:** Ensure `NULL` counts are below a 1% threshold for mandatory columns.
* **Uniqueness:** Check that Primary Keys are truly unique.
* **Range Checks:** Ensure numeric values (e.g., `Age`) fall within logical bounds (0-120).

In [ ]:
# Example: Basic Data Quality Audit with Pandas/Python
import pandas as pd

def audit_data(df):
    stats = {
        "total_rows": len(df),
        "missing_emails": df['email'].isnull().sum(),
        "duplicate_ids": df.duplicated(subset=['user_id']).sum(),
        "invalid_prices": (df['price'] < 0).sum()
    }
    
    # Trigger Alert if duplicates found
    if stats["duplicate_ids"] > 0:
        print(f"ALERT: {stats['duplicate_ids']} duplicates detected!")
    
    return stats

# In AWS, use 'AWS Glue Data Quality' to automate this without code.

<div style='padding:20px; background-color:#8e44ad; border-radius:10px; border-left: 10px solid #f39c12'>
<h1 style='color:white'>Big Data & AI Strategy</h1>
<p style='color:#f4ecf7'>Spark Tuning · Glue vs EMR · RAG Architecture · Engineering Leadership</p>
</div>

## 1. Spark Fundamentals & Shuffle Optimization

In Spark, the **Shuffle** is the process of redistributing data across the cluster. It is the most expensive operation because it involves Disk I/O and Network overhead.



### **Join Strategies:**
* **Broadcast Hash Join:** Best when one table is small (fits in memory). Spark sends the small table to every executor, avoiding a shuffle. 
* **Sort Merge Join:** The default for two large tables. It involves shuffling both tables based on the join key, sorting them, and then merging.

### **Performance Tuning:**
* **Salting:** Adding a random prefix to keys to fix **Data Skew** (where one partition is much larger than others).
* **Bucketing:** Pre-sorting and pre-partitioning data on disk to avoid shuffles in future joins.

## 2. AWS Glue vs. Amazon EMR

Both run Apache Spark, but they serve different operational needs.



| Feature | **AWS Glue** | **Amazon EMR** |
| :--- | :--- | :--- |
| **Management** | Serverless | Cluster-based (EC2) |
| **Startup Time** | Fast (seconds with G.1X/G.2X) | Slower (minutes to boot EC2s) |
| **Cost** | Pay per DPU-Hour | Pay per EC2 Instance Hour |
| **Customization** | Limited | Full control over Spark/Hadoop config |
| **Use Case** | Daily ETL, Batch Jobs | Long-running clusters, Interactive Data Science |

## 3. Generative AI: Retrieval Augmented Generation (RAG)

Data Engineers provide the "Context" for LLMs. Instead of retraining a model, we retrieve relevant documents from a **Vector Database** and pass them to the model (like Amazon Bedrock).



### **The Pipeline:**
1. **Chunking:** Break large PDFs/Docs into smaller pieces.
2. **Embedding:** Use an AI model to convert text into numerical vectors.
3. **Vector Store:** Save vectors in a DB like **OpenSearch** or **Pinecone**.
4. **Retrieval:** When a user asks a question, find the most similar vectors and send the text to the LLM.

## 4. Leadership: Mentoring & Design Reviews

Senior engineers are force multipliers. This involves:
* **Code Reviews:** Focus on performance (e.g., catching `SELECT *` or unnecessary shuffles).
* **Design Reviews:** Challenging assumptions about tool selection (e.g., "Why EMR instead of Glue?").
* **On-call Mentoring:** Creating runbooks to ensure the team can handle failures without the lead.